In [2]:
import pandas as pd
import numpy as np

import os

def load_hotel_reserve():
  customer_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/customer.csv')
  hotel_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/hotel.csv')
  reserve_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/reserve.csv')
  return customer_tb, hotel_tb, reserve_tb


def load_holiday_mst():
  holiday_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/holiday_mst.csv',
                           index_col=False)
  return holiday_tb


def load_production():
  production_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/production.csv')
  return production_tb


def load_production_missing_num():
  production_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/production_missing_num.csv')
  return production_tb


def load_production_missing_category():
  production_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/production_missing_category.csv')
  return production_tb


def load_monthly_index():
  monthly_index_tb = \
    pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/monthly_index.csv')
  return monthly_index_tb


def load_meros_txt():
  with open('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/txt/meros.txt', 'r') as f:
    meros = f.read()
    f.close()
  return meros


In [3]:
customer_tb, hotel_tb, reserve_tb = load_hotel_reserve()

In [ ]:
# 호텔 예약 레코드를 사용하여 성별을 불리언형과 범주형으로 변환

customer_tb[['sex_is_man']] = (customer_tb[['sex']] == 'man').astype('bool')    #성별이 남자인 경우 True, 아닌 경우 False

customer_tb['sex_c'] = \
    pd.Categorical(customer_tb['sex'], categories=['man', 'woman'])    #성별을 범주형으로

customer_tb['sex_c'].cat.codes    #범주형의 코드값을 확인
customer_tb['sex_c'].cat.categories    #범주형의 카테고리값을 확인

In [4]:
# 고객 테이블의 성별을 더미 변수로 만들기

customer_tb['sex']= pd.Categorical(customer_tb['sex'])  #성별을 범주형으로 변환

dummy_vars = pd.get_dummies(customer_tb['sex'], drop_first=False)    #성별을 더미 변수로 변환, drop_first=False로 설정하여 모든 범주에 대한 더미 변수를 생성

dummy_vars

,man,woman
0,True,False
1,True,False
2,False,True
3,True,False
4,True,False
...,...,...
995,True,False
996,True,False
997,False,True
998,False,True


In [6]:
# 범주그룹 단순화

customer_tb['age_rank'] = \
    pd.Categorical(np.floor(customer_tb['age']/10) * 10)    #고객의 나이를 10살 단위로 그룹화하여 범주형으로 변환

customer_tb['age_rank'] = customer_tb['age_rank'].cat.add_categories(['60이상'])   #60이상이라는 범주 추가

customer_tb.loc[customer_tb['age_rank'] \
                .isin( [60.0, 70.0, 80.0]), 'age_rank'] = '60이상'   #age_rank가 60.0, 70.0, 80.0인 경우 age_rank를 60이상으로 변경

customer_tb['age_rank'] = customer_tb['age_rank'].cat.remove_unused_categories()    #사용되지 않는 범주 제거   

customer_tb

,customer_id,age,sex,home_latitude,home_longitude,age_rank
0,c_1,41,man,35.092193,136.512347,40.0
1,c_2,38,man,35.325076,139.410551,30.0
2,c_3,49,woman,35.120543,136.511179,40.0
3,c_4,43,man,43.034868,141.240314,40.0
4,c_5,31,man,35.102661,136.523797,30.0
...,...,...,...,...,...,...
995,c_996,44,man,34.465648,135.373787,40.0
996,c_997,35,man,35.345372,139.413754,30.0
997,c_998,32,woman,43.062267,141.272126,30.0
998,c_999,48,woman,38.172800,140.464198,40.0


In [7]:
# 범주값의 보완 -KNN을 이용한 보완

production_missc_tb = load_production_missing_category()

from sklearn.neighbors import KNeighborsClassifier 

production_missc_tb.replace('None', np.nan, inplace=True)    #범주형 결측값인 'None'을 np.nan으로 변경

train = production_missc_tb.dropna(subset=['type'], inplace=False)   #type이 결측값이 아닌 레코드로 학습 데이터 생성

test = production_missc_tb \
    .loc[production_missc_tb.index.difference(train.index), :]   #train 데이터에 없는 레코드로 테스트 데이터 생성

kn = KNeighborsClassifier(n_neighbors=3)    #KNN 모델 생성, 이웃의 수는 3으로 설정

kn.fit(train[['length','thickness']], train['type'])   #train 데이터의 length와 thickness를 사용하여 type을 예측하는 모델 학습

test['type'] = kn.predict(test[['length','thickness']])   #test 데이터의 length와 thickness를 사용하여 type을 예측하여 test 데이터의 type 열에 저장